# Multi-drone multirotor control

Godot simulates the drones and publishes their state. This notebook computes formation targets, applies collision avoidance (boids), and publishes one set of motor forces for each active drone.

## Imports

In [43]:
import sys
sys.path.append("../../")

import math

from lib.data.dataplot import *
from lib.dds.dds import *
from lib.utils.time import *
from lib.system.controllers import *

## Constants

In [ ]:
# Constants
MIN_DRONES = 1
MAX_DRONES = 6
DEFAULT_DRONES = 4
RUN_SECONDS = 60

HOVER_ALTITUDE = 1.0
HOVER_FORCE = 2.5
MIN_FORCE = 0.0
MAX_FORCE = 8.0
FORMATION_SPACING = 1.4
TARGET_SPEED = 0.7
SAFE_DISTANCE = 0.8
AVOIDANCE_DISTANCE = 0.9
NEIGHBOR_DISTANCE = 2.0
LOOKAHEAD_TIME = 1.0
MAX_AVOIDANCE = 1.0
EPSILON = 1e-6

FORMATION_NONE = 0
FORMATION_LINE = 1
FORMATION_TRIANGLE = 2
FORMATION_SQUARE = 3
FORMATION_CIRCLE = 4
FORMATION_NAMES = ["NONE", "LINE", "TRIANGLE", "SQUARE", "CIRCLE"]

BOIDS_NONE = 0
BOIDS_SEPARATION = 1
BOIDS_PREDICTIVE = 2

STATE_TOPICS = {
    "X": "x", "Y": "y", "Z": "z",
    "TX": "roll", "TY": "pitch",
    "VX": "vx", "VY": "vy", "VZ": "vz",
    "WX": "roll_rate", "WY": "pitch_rate",
}
FORCE_TOPICS = ["f1", "f2", "f3", "f4"]
TARGET_TOPICS = ["target_point_x", "target_point_y", "target_point_z"]
DEBUG_FLOAT_TOPICS = [
    "boids_x", "boids_y", "boids_length", "boids_safe_distance",
    "base_target_x", "base_target_y", "base_target_z",
    "target_x", "target_y", "target_z", "target_distance",
]

# Helper Functions

In [ ]:
# Utility functions

def topic(drone_id, name):
    return f"D{drone_id}_{name}"

def clamp(value, minimum, maximum):
    return max(minimum, min(value, maximum))

# Euclidean length of a 2D vector: sqrt(x^2 + y^2).
def length_2d(vector):
    return math.hypot(vector[0], vector[1])

# Normalize a non-zero 2D vector to unit length.
def normalize_2d(vector):
    length = length_2d(vector)
    if length < EPSILON:
        return (0.0, 0.0)
    return (vector[0] / length, vector[1] / length)

# Limits the length of a 2D vector to a maximum value.
def limit_2d(vector, maximum):
    length = length_2d(vector)
    if length <= maximum:
        return vector
    scale = maximum / length
    return (vector[0] * scale, vector[1] * scale)


# Calculates the Euclidean distance between two 3D points.
# formula: sqrt((x2-x1)^2 + (y2-y1)^2 + (z2-z1)^2).
def distance_3d(first, second):
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(first, second)))


def state_position(state):
    return (state["x"], state["y"], state["z"])


# Subtract the mean X/Y offset so the formation centroid stays at the origin.
def center_offsets(offsets):
    center_x = sum(offset[0] for offset in offsets) / len(offsets)
    center_y = sum(offset[1] for offset in offsets) / len(offsets)
    return [(x - center_x, y - center_y, z) for x, y, z in offsets]

# Calculate local offsets for a formation and number of drones.
def formation_offsets(drone_count, formation_id):
    if drone_count == 0:
        return []

    if formation_id == FORMATION_LINE:
        offsets = []
        center = (drone_count - 1) / 2
        for i in range(drone_count):
            x = (i - center) * FORMATION_SPACING
            offsets.append((x, 0.0, 0.0))
        return offsets

    if formation_id == FORMATION_TRIANGLE:
        # Triangular rows contain 1, 2, 3, ... equally spaced drones.
        offsets = []
        row = 1
        while len(offsets) < drone_count:
            y = (row - 1) * FORMATION_SPACING
            for column in range(row):
                x = (column - (row - 1) / 2) * FORMATION_SPACING
                offsets.append((x, y, 0.0))
                if len(offsets) == drone_count:
                    break
            row += 1
        return center_offsets(offsets)

    if formation_id == FORMATION_SQUARE:
        # ceil(sqrt(n)) columns produce the smallest near-square grid.
        columns = math.ceil(math.sqrt(drone_count))
        rows = math.ceil(drone_count / columns)
        offsets = []
        for i in range(drone_count):
            column = i % columns
            row = i // columns
            x = (column - (columns - 1) / 2) * FORMATION_SPACING
            y = (row - (rows - 1) / 2) * FORMATION_SPACING
            offsets.append((x, y, 0.0))
        return center_offsets(offsets)

    if formation_id == FORMATION_CIRCLE and drone_count > 1:
        # Divide 2*pi uniformly so every drone receives an equal angular step.
        offsets = []
        for i in range(drone_count):
            angle = 2 * math.pi * i / drone_count
            x = FORMATION_SPACING * math.cos(angle)
            y = FORMATION_SPACING * math.sin(angle)
            offsets.append((x, y, 0.0))
        return offsets

    # NONE keeps the drones in the same line used for their initial spawn positions.
    offsets = []
    center = (drone_count - 1) / 2
    for i in range(drone_count):
        y = (i - center) * FORMATION_SPACING
        offsets.append((0.0, y, 0.0))
    return offsets

# Translate every local formation offset into an absolute target position.
# The three components are X, Y and Z: slot[i] = origin[i] + offset[i].
def formation_slots(drone_count, formation_id, origin):
    slots = []
    for offset in formation_offsets(drone_count, formation_id):
        x = origin[0] + offset[0]
        y = origin[1] + offset[1]
        z = origin[2] + offset[2]
        slots.append((x, y, z))

    return slots

# Greedily assign the nearest remaining drone-slot pair to reduce travel distance.
def nearest_slot_assignment(active_ids, slots, states):
    # Start with all drones and slots still available.
    remaining_ids = set(active_ids)
    remaining_slots = set(range(len(slots)))
    assignment = {}

    # Repeat until every drone has a slot.
    while remaining_ids:
        # Try every remaining drone-slot pair.
        possible_pairs = []
        for drone_id in sorted(remaining_ids):
            for slot_id in sorted(remaining_slots):
                drone_position = state_position(states[drone_id])
                slot_position = slots[slot_id]
                distance = distance_3d(drone_position, slot_position)
                # Put distance first so min() chooses the closest pair.
                possible_pairs.append((distance, drone_id, slot_id))

        # Read the drone and slot IDs from the closest pair.
        closest_pair = min(possible_pairs)
        drone_id = closest_pair[1]
        slot_id = closest_pair[2]

        # Save the match and remove both items from the available sets.
        assignment[drone_id] = slot_id
        remaining_ids.remove(drone_id)
        remaining_slots.remove(slot_id)

    return assignment

# Moves the current target position towards the desired position.
def move_target(current, desired, maximum_step):
    delta = (desired[0] - current[0], desired[1] - current[1])
    if length_2d(delta) <= maximum_step:
        return desired
    direction = normalize_2d(delta)
    return (
        current[0] + direction[0] * maximum_step,
        current[1] + direction[1] * maximum_step,
        desired[2],
    )

# Manages the target positions for drones in a formation, updating them based on the formation configuration and drone states.
class FormationTargets:
    def __init__(self):
        self.signature = None
        self.assignment = {}
        self.targets = {}

    def update(self, active_ids, formation_id, origin, states, delta_t):
        signature = (tuple(active_ids), formation_id)
        slots = formation_slots(len(active_ids), formation_id, origin)

        if signature != self.signature:
            self.assignment = nearest_slot_assignment(active_ids, slots, states)
            self.targets = {drone_id: state_position(states[drone_id]) for drone_id in active_ids}
            self.signature = signature

        maximum_step = max(delta_t, EPSILON) * TARGET_SPEED
        for drone_id in active_ids:
            desired = slots[self.assignment[drone_id]]
            self.targets[drone_id] = move_target(self.targets[drone_id], desired, maximum_step)

        self.targets = {drone_id: self.targets[drone_id] for drone_id in active_ids}
        return dict(self.targets)


## Multirotor controller

Each drone owns a separate controller because every PID keeps its own internal error history.

In [ ]:
# Multirotor Class
class Multirotor:
    def __init__(self):
        self.vz_control = PID_Controller(5.0, 10.0, 0.0, 5)
        self.z_control = PID_Controller(2.0, 0.0, 0.0, 2)

        self.w_roll_control = PID_Controller(0.75, 0.3, 0.0075, 2)
        self.roll_control = PID_Controller(1.0, 0.0, 0.0, 2)
        self.w_pitch_control = PID_Controller(0.75, 0.3, 0.0075, 2)
        self.pitch_control = PID_Controller(1.0, 0.0, 0.0, 2)

        self.vy_control = PID_Controller(0.4, 0.01, 0.25, math.radians(30))
        self.y_control = PID_Controller(1.0, 0.0, 0.0, 2.0)
        self.vx_control = PID_Controller(0.4, 0.01, 0.25, math.radians(30))
        self.x_control = PID_Controller(1.0, 0.0, 0.0, 2.0)

        self.target = (0.0, 0.0, HOVER_ALTITUDE)
        self.vx_target = 0.0
        self.vy_target = 0.0
        self.vz_target = 0.0

    # Cascaded feedback: position errors become velocity targets, then attitude
    # and angular-rate targets, and finally corrections to the motor forces.
    def evaluate(self, delta_t, state, target):
        self.target = target

        # Outer altitude loop sets vertical speed; the inner loop corrects thrust.
        self.vz_target = self.z_control.evaluate(delta_t, target[2] - state["z"])
        altitude = self.vz_control.evaluate(delta_t, self.vz_target - state["vz"])
        base_force = clamp(HOVER_FORCE + altitude, MIN_FORCE, MAX_FORCE)

        # Motion along Y is obtained by roll; the minus sign follows the axis convention.
        self.vy_target = self.y_control.evaluate(delta_t, target[1] - state["y"])
        roll_target = -self.vy_control.evaluate(delta_t, self.vy_target - state["vy"])
        roll_rate_target = self.roll_control.evaluate(delta_t, roll_target - state["roll"])
        roll = self.w_roll_control.evaluate(delta_t, roll_rate_target - state["roll_rate"])

        # Motion along X is obtained by commanding pitch.
        self.vx_target = self.x_control.evaluate(delta_t, target[0] - state["x"])
        pitch_target = self.vx_control.evaluate(delta_t, self.vx_target - state["vx"])
        pitch_rate_target = self.pitch_control.evaluate(delta_t, pitch_target - state["pitch"])
        pitch = self.w_pitch_control.evaluate(delta_t, pitch_rate_target - state["pitch_rate"])

        # Propeller layout:
        #   3   4
        #   2   1
        # All motors share the altitude force. 
        # Roll changes the 1/4 and 2/3 pairs. 
        # Pitch changes the 3/4 and 1/2 pairs.
        forces = (
            base_force + roll - pitch,
            base_force - roll - pitch,
            base_force - roll + pitch,
            base_force + roll + pitch,
        )
        return tuple(clamp(force, MIN_FORCE, MAX_FORCE) for force in forces)


## Boids: Collision avoidance

Separation pushes apart drones that are already too close. Predictive avoidance checks where each nearby pair is heading and moves their targets sideways if they may collide. These rules only change target positions; the PID controller still calculates the motor forces.

In [ ]:
# Collision Avoidance
# Point toward the target without moving faster than TARGET_SPEED.
def planned_velocity(state, target):
    delta = (target[0] - state["x"], target[1] - state["y"])
    direction = normalize_2d(delta)
    speed = min(TARGET_SPEED, length_2d(delta))
    return (direction[0] * speed, direction[1] * speed)

# Move targets to keep nearby drones from colliding.
def collision_avoidance(states, base_targets):
    offsets = {}
    modes = {}
    for drone_id in states:
        offsets[drone_id] = (0.0, 0.0)
        modes[drone_id] = BOIDS_NONE

    active_ids = sorted(states)

    # Check every pair once.
    for first_index in range(len(active_ids)):
        first_id = active_ids[first_index]
        for second_index in range(first_index + 1, len(active_ids)):
            second_id = active_ids[second_index]
            first_state = states[first_id]
            second_state = states[second_id]

            relative_x = second_state["x"] - first_state["x"]
            relative_y = second_state["y"] - first_state["y"]
            relative_position = (relative_x, relative_y)
            current_distance = length_2d(relative_position)

            # Optimization: Far drones cannot affect each other.
            if current_distance > NEIGHBOR_DISTANCE:
                continue

            # Push apart drones that are already too close.
            if current_distance < AVOIDANCE_DISTANCE:
                away = normalize_2d((-relative_position[0], -relative_position[1]))
                if current_distance < EPSILON:
                    away = (1.0, 0.0)

                strength = (AVOIDANCE_DISTANCE - current_distance) / AVOIDANCE_DISTANCE
                correction = (away[0] * 1.6 * strength, away[1] * 1.6 * strength)
                pair_mode = BOIDS_SEPARATION

            # Otherwise check whether the pair is heading toward a collision.
            else:
                first_velocity = planned_velocity(first_state, base_targets[first_id])
                second_velocity = planned_velocity(second_state, base_targets[second_id])
                relative_velocity = (
                    second_velocity[0] - first_velocity[0],
                    second_velocity[1] - first_velocity[1],
                )
                speed_squared = relative_velocity[0] ** 2 + relative_velocity[1] ** 2

                # No relative movement means no predicted collision.
                if speed_squared < EPSILON:
                    continue

                # Find the pair's closest point during the look-ahead time.
                dot_product = (
                    relative_position[0] * relative_velocity[0]
                    + relative_position[1] * relative_velocity[1]
                )
                closest_time = clamp(-dot_product / speed_squared, 0.0, LOOKAHEAD_TIME)
                closest_vector = (
                    relative_position[0] + relative_velocity[0] * closest_time,
                    relative_position[1] + relative_velocity[1] * closest_time,
                )
                closest_distance = length_2d(closest_vector)

                # Skip pairs that stay safe or move farther apart.
                if closest_distance >= AVOIDANCE_DISTANCE or closest_distance >= current_distance:
                    continue

                # Push away and slightly sideways to avoid the collision.
                away = normalize_2d((-closest_vector[0], -closest_vector[1]))
                if closest_distance < EPSILON:
                    away = normalize_2d((-relative_position[0], -relative_position[1]))

                side = (-away[1], away[0])
                strength = (AVOIDANCE_DISTANCE - closest_distance) / AVOIDANCE_DISTANCE
                correction = (
                    (away[0] * 0.8 + side[0] * 0.35) * strength,
                    (away[1] * 0.8 + side[1] * 0.35) * strength,
                )
                pair_mode = BOIDS_PREDICTIVE

            # Apply the same correction in opposite directions.
            first_offset = offsets[first_id]
            second_offset = offsets[second_id]
            offsets[first_id] = (
                first_offset[0] + correction[0],
                first_offset[1] + correction[1],
            )
            offsets[second_id] = (
                second_offset[0] - correction[0],
                second_offset[1] - correction[1],
            )

            # Direct separation has priority over predictive avoidance.
            if pair_mode == BOIDS_SEPARATION:
                modes[first_id] = BOIDS_SEPARATION
                modes[second_id] = BOIDS_SEPARATION
            else:
                if modes[first_id] == BOIDS_NONE:
                    modes[first_id] = BOIDS_PREDICTIVE
                if modes[second_id] == BOIDS_NONE:
                    modes[second_id] = BOIDS_PREDICTIVE

    # Limit each offset and use it to build the final target.
    adjusted_targets = {}
    for drone_id, base_target in base_targets.items():
        offset = limit_2d(offsets[drone_id], MAX_AVOIDANCE)
        offsets[drone_id] = offset

        if modes[drone_id] == BOIDS_SEPARATION:
            state = states[drone_id]
            target_x = state["x"] + offset[0]
            target_y = state["y"] + offset[1]
        else:
            target_x = base_target[0] + offset[0]
            target_y = base_target[1] + offset[1]

        adjusted_targets[drone_id] = (target_x, target_y, base_target[2])

    return adjusted_targets, offsets, modes


## DDS control loop

In [ ]:
# MAIN

# Build the list of DDS topics needed by the notebook.
def subscribe_topics():
    topics = ["start", "tick", "current_drones", "drone_count", "formation_id"] + TARGET_TOPICS
    for drone_id in range(MAX_DRONES):
        topics.append(topic(drone_id, "active"))
        topics.extend(topic(drone_id, name) for name in STATE_TOPICS)
    return topics

# Read one DDS value, or use a default when no value is available.
def read_value(dds, name, default=0.0):
    value = dds.read(name)
    return default if value is None else value

# Read the requested drone count and keep it inside the allowed limits.
def read_drone_count(dds):
    count = int(read_value(dds, "current_drones", read_value(dds, "drone_count", DEFAULT_DRONES)))
    return int(clamp(count, MIN_DRONES, MAX_DRONES))

# Read the selected formation and keep its ID valid.
def read_formation(dds):
    return int(clamp(int(read_value(dds, "formation_id", FORMATION_NONE)), FORMATION_NONE, FORMATION_CIRCLE))

# Read the X, Y and Z origin shared by the formation.
def read_origin(dds):
    return tuple(read_value(dds, name, default) for name, default in zip(TARGET_TOPICS, (0.0, 0.0, HOVER_ALTITUDE)))

# Read all state values for one drone.
def read_state(dds, drone_id):
    return {key: read_value(dds, topic(drone_id, name)) for name, key in STATE_TOPICS.items()}

# Send the four motor forces to one drone.
def publish_forces(dds, drone_id, forces):
    for name, force in zip(FORCE_TOPICS, forces):
        dds.publish(topic(drone_id, name), force, DDS.DDS_TYPE_FLOAT)

# Stop one drone and reset all of its debug values.
def clear_drone(dds, drone_id):
    publish_forces(dds, drone_id, (0.0, 0.0, 0.0, 0.0))
    for name in ["boids_active", "boids_neighbors", "boids_mode"]:
        dds.publish(topic(drone_id, name), 0, DDS.DDS_TYPE_INT)
    for name in DEBUG_FLOAT_TOPICS:
        dds.publish(topic(drone_id, name), 0.0, DDS.DDS_TYPE_FLOAT)

# Count how many drones are close to the selected drone.
def neighbor_count(drone_id, states):
    position = state_position(states[drone_id])
    return sum(
        distance_3d(position, state_position(other)) <= NEIGHBOR_DISTANCE
        for other_id, other in states.items()
        if other_id != drone_id
    )

# Return the distance between the closest pair of drones.
def minimum_distance(states):
    ids = sorted(states)
    distances = [
        distance_3d(state_position(states[first]), state_position(states[second]))
        for index, first in enumerate(ids)
        for second in ids[index + 1:]
    ]
    return min(distances) if distances else None

# Send collision-avoidance and target values to the Godot debug display.
def publish_debug(dds, drone_id, state, base_target, target, offset, mode, neighbors):
    integer_values = {"boids_active": 1, "boids_neighbors": neighbors, "boids_mode": mode}
    float_values = {
        "boids_x": offset[0], "boids_y": offset[1], "boids_length": length_2d(offset),
        "boids_safe_distance": SAFE_DISTANCE,
        "base_target_x": base_target[0], "base_target_y": base_target[1], "base_target_z": base_target[2],
        "target_x": target[0], "target_y": target[1], "target_z": target[2],
        "target_distance": distance_3d(state_position(state), target),
    }
    for name, value in integer_values.items():
        dds.publish(topic(drone_id, name), value, DDS.DDS_TYPE_INT)
    for name, value in float_values.items():
        dds.publish(topic(drone_id, name), value, DDS.DDS_TYPE_FLOAT)

# Create one plot with a shared time axis and the requested signals.
def create_plotter(series):
    plotter = DataPlotter()
    plotter.set_x("time (seconds)")
    for name, label in series:
        plotter.add_y(name, label)
    return plotter

# Create all position, velocity, attitude and error plots for one drone.
def create_drone_plotters(drone_id):
    prefix = f"D{drone_id}"
    plotters = {
        f"position_{axis}": create_plotter([
            (f"target_{axis}", f"{prefix} target {axis.upper()}"),
            (f"current_{axis}", f"{prefix} current {axis.upper()}"),
        ])
        for axis in "xyz"
    }
    plotters["velocity"] = create_plotter([
        ("vx", f"{prefix} VX"),
        ("vy", f"{prefix} VY"),
        ("vz", f"{prefix} VZ"),
    ])
    plotters["attitude"] = create_plotter([
        ("roll", f"{prefix} roll"),
        ("pitch", f"{prefix} pitch"),
    ])
    plotters["error"] = create_plotter([("target_error", f"{prefix} target error")])
    return plotters

# Add one state-and-target sample to all plots of a drone.
def append_drone_plots(plotters, current_time, state, target):
    for axis_id, axis in enumerate("xyz"):
        plotter = plotters[f"position_{axis}"]
        plotter.append_x(current_time)
        plotter.append_y(f"target_{axis}", target[axis_id])
        plotter.append_y(f"current_{axis}", state[axis])

    velocity_plot = plotters["velocity"]
    velocity_plot.append_x(current_time)
    for axis in "xyz":
        velocity_plot.append_y(f"v{axis}", state[f"v{axis}"])

    attitude_plot = plotters["attitude"]
    attitude_plot.append_x(current_time)
    attitude_plot.append_y("roll", state["roll"])
    attitude_plot.append_y("pitch", state["pitch"])

    error_plot = plotters["error"]
    error_plot.append_x(current_time)
    error_plot.append_y("target_error", distance_3d(state_position(state), target))

# Compare the closest drone pair with the configured safety distance.
swarm_distance_plot = create_plotter([
    ("minimum_distance", "minimum drone distance"),
    ("safe_distance", "safe distance"),
])


controllers = {}
drone_plotters = {}
formation_targets = FormationTargets()
previous_active_ids = set()
previous_formation = None
minimum_distance_seen = None

dds = DDS()
dds.start()
dds.subscribe(subscribe_topics())

print("Waiting for Godot")
dds.wait("start")
print("Started")

t = Time()
t.start()

try:
    # Process simulator ticks until the experiment time is over.
    while t.get() < RUN_SECONDS:
        dds.wait("tick")
        delta_t = t.elapsed()

        # Ignore ticks that arrive too close together.
        if delta_t <= EPSILON:
            continue

        # Read the current swarm settings.
        drone_count = read_drone_count(dds)
        formation_id = read_formation(dds)
        origin = read_origin(dds)

        # Keep only active drones and read their states.
        active_ids = []
        for drone_id in range(drone_count):
            active = int(read_value(dds, topic(drone_id, "active"), 1))
            if active == 1:
                active_ids.append(drone_id)

        states = {}
        for drone_id in active_ids:
            states[drone_id] = read_state(dds, drone_id)

        if formation_id != previous_formation:
            print("Formation:", FORMATION_NAMES[formation_id])
            previous_formation = formation_id

        # Build formation targets, then move unsafe targets away from collisions.
        base_targets = formation_targets.update(active_ids, formation_id, origin, states, delta_t)
        targets, offsets, modes = collision_avoidance(states, base_targets)

        # Update each controller and publish its outputs.
        for drone_id in active_ids:
            if drone_id not in controllers:
                controllers[drone_id] = Multirotor() # Check for newly spawned drones

            controller = controllers[drone_id]
            forces = controller.evaluate(delta_t, states[drone_id], targets[drone_id])
            publish_forces(dds, drone_id, forces)

            neighbors = neighbor_count(drone_id, states)
            publish_debug(
                dds,
                drone_id,
                states[drone_id],
                base_targets[drone_id],
                targets[drone_id],
                offsets[drone_id],
                modes[drone_id],
                neighbors,
            )

            if drone_id not in drone_plotters:
                drone_plotters[drone_id] = create_drone_plotters(drone_id)
            append_drone_plots(drone_plotters[drone_id], t.get(), states[drone_id], targets[drone_id])

        # Stop drones that were active during the previous tick but are now disabled.
        current_active_ids = set(active_ids)
        inactive_ids = previous_active_ids - current_active_ids
        for drone_id in inactive_ids:
            clear_drone(dds, drone_id)
            controllers.pop(drone_id, None)
        previous_active_ids = current_active_ids

        # Record the closest distance measured during the experiment.
        current_minimum = minimum_distance(states)
        if current_minimum is not None:
            swarm_distance_plot.append_x(t.get())
            swarm_distance_plot.append_y("minimum_distance", current_minimum)
            swarm_distance_plot.append_y("safe_distance", SAFE_DISTANCE)

            if minimum_distance_seen is None:
                minimum_distance_seen = current_minimum
            else:
                minimum_distance_seen = min(minimum_distance_seen, current_minimum)

finally:
    # Always stop every drone and close DDS, even if the loop fails.
    for drone_id in range(MAX_DRONES):
        clear_drone(dds, drone_id)
    dds.stop()

for drone_id in sorted(drone_plotters):
    plot_multiple(list(drone_plotters[drone_id].values()), figsize=(14, 16))
swarm_distance_plot.plot()
if minimum_distance_seen is not None:
    print(f"Minimum inter-drone distance: {minimum_distance_seen:.3f} m")
print("Done")


Waiting for Godot
Started
Formation: LINE


KeyboardInterrupt: 